# ALOHA Prepared 结果查看

用于 `prepare_dataset.py` 完成后查看保留的数据。只需填写 Prepared 表名（namespace 默认 default），随后在下拉框选择 episode。

本入口不需要 Raw 或标注文件，不生成 Prepared、不修改数据、不提交 Ray Job。能展示保留帧及内部缺口，不能仅凭 Prepared 确定原轨迹被删掉的起始/末尾帧、整条被删除的 episode 或删除原因。

表中已有相机的图像（支持四路，兼容两路）、state/action、split 和有效区间直接来自 Prepared。使用已有 Notebook 通道，无额外服务端口。每次加载一条 episode 到内存，拖动滑块不会重新查库。

相机通过第 1 步的 `CAMERAS` 选择，默认四路，与 import_raw.py 一致；只查询和显示所选相机，以两列排列。两路表请填写实际存在的相机。这里只支持逐帧 JPEG 图片列，不直接解码 GOP 表。

**表名来源：** `PREPARED_TABLE` 对应 `prepare_dataset.py` 的 `--prepared-table`，本教程统一为 `aloha_velcro_prepared_v1`。Raw 标注 Notebook 不创建此表；必须先成功运行 Prepare。若你的 Prepare 命令使用其他表名，请填写那个实际输出表名，namespace 也必须一致。


## 第 1 步：填写 Prepared 表名和相机
更换表或相机后重新运行第 1～4 步，再点击加载。

- 四路：保留默认 `CAMERAS`。
- 一路：`CAMERAS = ["cam_high"]`。
- 两路：`CAMERAS = ["cam_high", "cam_left_wrist"]`。
- 不读图片：`CAMERAS = []`，只查看 state/action 和帧信息。

未选相机不会进入数据库查询，不只是隐藏画面。每次仍将所选 Episode 的全部帧加载到内存；减少相机可降低图片内存占用，改变显示宽度不会降低已读取 JPEG 的内存。


In [ ]:
import html
import os

import ipywidgets as widgets
from daft import col
from IPython.display import display
from lakesoul.catalog import LakeSoulCatalog

NAMESPACE = "default"
PREPARED_TABLE = "aloha_velcro_prepared_v1"  # 与 prepare_dataset.py 的 --prepared-table 一致


# 支持的相机名称；CAMERAS 决定实际读取哪些图片列。
CAMERA_NAMES = ("cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist")
CAMERAS = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]


def selected_cameras(schema_names):
    # 空列表用于只查看向量；拼写错误或缺列必须报错，不能静默少读一路。
    if not isinstance(CAMERAS, (list, tuple)) or any(not isinstance(name, str) for name in CAMERAS):
        raise ValueError('CAMERAS 必须是相机名称列表，例如 ["cam_high"]')
    if len(set(CAMERAS)) != len(CAMERAS):
        raise ValueError("CAMERAS 中不能有重复相机")
    unsupported = set(CAMERAS) - set(CAMERA_NAMES)
    missing = set(CAMERAS) - set(schema_names)
    if unsupported or missing:
        raise ValueError(f"不支持的相机：{sorted(unsupported)}；表中缺少的相机：{sorted(missing)}")
    return list(CAMERAS)


def make_camera_grid(width=360):
    images = {name: widgets.Image(format="jpeg", width=width) for name in CAMERA_NAMES}
    panels = {name: widgets.VBox([widgets.HTML(f"<b>{name}</b>"), images[name]],
                                layout=widgets.Layout(display="none"))
              for name in CAMERA_NAMES}
    grid = widgets.GridBox(list(panels.values()), layout=widgets.Layout(
        grid_template_columns=f"repeat(2, {width}px)", grid_gap="8px"))
    return images, panels, grid


def update_camera_images(images, panels, row=None):
    # 无此列则隐藏；清空或切换样本时同时清掉旧图片。
    row = {} if row is None else row
    for name, image in images.items():
        image.value = bytes(row.get(name) or b"")
        panels[name].layout.display = "" if name in row else "none"


## 第 2 步：连接 LakeSoul
沿用 PG/S3 环境变量。


In [ ]:
endpoint = os.getenv("AWS_ENDPOINT") or os.getenv("AWS_ENDPOINT_URL")

object_store_options = {
    "fs.s3a.path.style.access": "true",
}

if endpoint:
    object_store_options["fs.s3a.endpoint"] = endpoint

if os.getenv("AWS_ACCESS_KEY_ID"):
    object_store_options["fs.s3a.access.key"] = os.environ["AWS_ACCESS_KEY_ID"]

if os.getenv("AWS_SECRET_ACCESS_KEY"):
    object_store_options["fs.s3a.secret.key"] = os.environ["AWS_SECRET_ACCESS_KEY"]

catalog = LakeSoulCatalog.from_env(
    namespace=NAMESPACE,
    object_store_options=object_store_options,
)
print("Catalog 已创建，命名空间：", NAMESPACE)


## 第 3 步：列出可查看的 Episode
仅列举 Prepared 中实际存在的轨迹；被全部剔除的轨迹不会出现在列表里。


In [ ]:
# 只读取 episode_id 列，不在列举轨迹时读取图像。
if not PREPARED_TABLE.strip():
    raise ValueError("请先在第 1 步填写 Prepared 表名并运行该单元格")
prepared_context = (NAMESPACE, PREPARED_TABLE)
episode_ids = (
    catalog.table(PREPARED_TABLE).scan(columns=["episode_id"]).to_daft()
    .select("episode_id").distinct().sort("episode_id").to_arrow().to_pylist()
)
print("Prepared 表：", PREPARED_TABLE, "；Episode 数：", len(episode_ids))


## 第 4 步：选择 Episode 并加载
点击“加载所选轨迹”后可播放表中已有相机的画面、查看 state/action。显示原始帧号、split、有效区间；遇到缺帧或区间变化会提示。每 100 毫秒浏览一帧，不是原始 50 FPS。切换下拉框后再次点击加载。


In [ ]:
# 重新运行这一格时，先停掉旧播放器。
if globals().get("play") is not None:
    play.playing = False
if globals().get("frame_link") is not None:
    frame_link.unlink()
if globals().get("viewer") is not None:
    viewer.close()

play = widgets.Play(value=0, min=0, max=0, step=1, interval=100, disabled=True)
frame_slider = widgets.IntSlider(
    value=0, min=0, max=0, step=1,
    description="帧位置：", continuous_update=False,
    disabled=True, layout=widgets.Layout(width="75%"),
)
frame_link = widgets.jslink((play, "value"), (frame_slider, "value"))
frame_info = widgets.HTML()
camera_images, camera_panels, camera_grid = make_camera_grid()
state_output = widgets.HTML()
action_output = widgets.HTML()
episode_rows = []


def format_vector(value):
    if value is None:
        return "null"
    return "[" + ", ".join(f"{float(item):.5f}" for item in value) + "]"


def render_frame(position):
    if not episode_rows:
        return
    row = episode_rows[position]
    timestamp = row.get("timestamp")
    timestamp_text = "null" if timestamp is None else f"{float(timestamp):.4f} s"
    notes = []
    if position > 0:
        previous = episode_rows[position - 1]
        if row["frame_index"] != previous["frame_index"] + 1:
            notes.append("帧号不连续：这里有缺口或重复帧")
        if row.get("valid_interval_id") != previous.get("valid_interval_id"):
            notes.append("进入另一个有效区间，训练窗口不能跨此边界")

    table_name = prepared_context[1]
    frame_info.value = (
        f"<b>表：</b>{html.escape(table_name)}　"
        f"<b>Episode：</b>{html.escape(str(row['episode_id']))}　"
        f"<b>frame_index：</b>{row['frame_index']}　"
        f"<b>timestamp：</b>{timestamp_text}　"
        f"<b>位置：</b>{position + 1}/{len(episode_rows)}<br>"
        f"<b>split：</b>{html.escape(str(row['split']))}　"
        f"<b>有效区间：</b>{html.escape(str(row['valid_interval_id']))}<br>"
        f"<strong>{html.escape('；'.join(notes))}</strong>"
    )
    update_camera_images(camera_images, camera_panels, row)
    state_output.value = "<b>observation_state</b><pre>" + html.escape(format_vector(row.get("observation_state"))) + "</pre>"
    action_output.value = "<b>action</b><pre>" + html.escape(format_vector(row.get("action"))) + "</pre>"


def load_episode(change=None):
    global episode_rows
    play.playing = False
    play.disabled = frame_slider.disabled = True
    # 先清空数据，避免旧滑块位置触发越界回调。
    episode_rows = []
    play.value = frame_slider.value = 0
    update_camera_images(camera_images, camera_panels)
    state_output.value = action_output.value = ""
    if prepared_context != (NAMESPACE, PREPARED_TABLE):
        frame_info.value = "表配置已改变，请重新执行第 2～4 步。"
        return
    if episode_choice.value is not None:
        frame_info.value = "正在读取所选 Episode…"
        try:
            table = catalog.table(prepared_context[1])
            cameras = selected_cameras(table.schema.names)
            episode_rows = (
                table.scan(columns=[
                    "episode_id", "frame_index", "timestamp", "observation_state", "action",
                    *cameras, "split", "valid_interval_id",
                ]).to_daft()
                .where(col("episode_id") == episode_choice.value)
                .sort("frame_index").to_arrow().to_pylist()
            )
        except Exception as error:
            frame_info.value = "读取失败：" + html.escape(str(error))
            return
    if not episode_rows:
        play.max = frame_slider.max = 0
        frame_info.value = "该表中没有此 Episode 的记录，请检查筛选结果或表名。"
        return
    play.max = frame_slider.max = len(episode_rows) - 1
    play.disabled = frame_slider.disabled = False
    render_frame(0)


episode_choice = widgets.Dropdown(options=[r["episode_id"] for r in episode_ids], description="Episode：")
load_button = widgets.Button(description="加载所选轨迹", disabled=not episode_ids)
load_button.on_click(load_episode)
frame_slider.observe(lambda change: render_frame(change["new"]), names="value")
viewer = widgets.VBox([
    widgets.HBox([episode_choice, load_button]),
    widgets.HBox([play, frame_slider]),
    frame_info,
    camera_grid,
    state_output,
    action_output,
])
frame_info.value = "选择 Episode 后点击加载。" if episode_ids else "Prepared 表没有记录。"
display(viewer)
